# Citation reliability of a small open-weight LLM
**Question:** when asked for references, does a small model return papers that exist, with links that work, and does that rate drop for niche or recent topics?


| Step | What happens |
|---|---|
| 1-2 | Install libraries, check GPU |
| 3 | Configuration: model, topics |
| 4 | Build `prompts.jsonl` |
| 5-6 | Load the model, generate with greedy decoding |
| 7-8 | Verify every reference against Crossref and the live link |
| 9 | Statistics (Wilson CI, Fisher exact test) |


## Step 1 - Install libraries

In [1]:
!pip -q install transformers accelerate requests scipy pandas

## Step 2 - Imports and GPU check

In [2]:
import csv, json, math, os, random, re, time
from collections import Counter
from difflib import SequenceMatcher
from functools import lru_cache

import pandas as pd
import requests
import torch
from scipy.stats import fisher_exact

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: in Colab use Runtime > Change runtime type > T4 GPU")

CUDA available: True
GPU: Tesla T4


## Step 3 - Configuration  

  * **A** - famous, heavily cited papers/topics (control: the model has probably seen them many times)
  * **B** - niche sub-topics from your own field
  * **C** - topics newer than the model's training data (check the model card for its cutoff)

In [ ]:
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"       
TRUST_REMOTE_CODE = False            
MAILTO = "ayansaleem827@gmail.com"           

N_REFS = 5                           
MAX_NEW_TOKENS = 900                 
SEED = 0

TEMPLATES = [
    "List {n} real research papers about: {topic}. For each give title, authors, year, venue, doi and url.",
    "I am writing a literature review on {topic}. Give me {n} references I can cite, with title, authors, year, venue, doi and url.",
]
FORMAT_INSTRUCTION = (
    " Return ONLY a JSON list of objects with the keys "
    "title, authors, year, venue, doi, url. No other text."
)

TOPICS = {
    "A": [
        "Transformer",
        "GANs",
        "ResNet",
        "BERT",
        "GPT-3",
        "Vision Transformer",
        "CLIP",
        "Diffusion models",
        "AlphaFold",
        "RLHF",
        "Dropout",
        "Batch normalization",
        "Adam optimizer",
        "Word2Vec",
        "Seq2Seq",
],
    "B": [  
        "Lighting-robust open semantic mapping",
        "Low-light semantic mapping for indoor robots",
        "Illumination-invariant place recognition",
        "Appearance-robust semantic segmentation",
        "Shadow-aware open-vocabulary mapping",
        "Underwater monocular odometry",
        "Underwater depth estimation for navigation",
        "Refractive distortion correction underwater",
        "Underwater visual-inertial odometry",
        "Low-texture underwater visual odometry",
        "Rigid-body constrained 4D Gaussian splatting",
        "Dynamic Gaussian splatting for robot mapping",
        "Gaussian splatting with rigid motion priors",
        "4D Gaussian splatting for robot localisation",
        "Gaussian splatting for deformable scene reconstruction",
    ],
    "C": [ 
        "Event-based Gaussian splatting SLAM",
        "Neural radiance field place recognition",
        "Radiance field localisation for autonomous driving",
        "Vision-language-action models for mobile manipulation",
        "World models for embodied agents",
        "Diffusion policies for legged locomotion",
        "Sim-to-real transfer for humanoid robots",
        "Distributed multi-agent SLAM",
        "Tactile-informed robot manipulation policies",
        "Continual learning for robot policies",
        "Uncertainty-aware robot perception",
        "Language-grounded robot navigation",
        "3D foundation models for robot perception",
        "Embodied question answering with large language models",
        "Foundation models for underwater robotics",
    ],
}

## Step 4 - Build `prompts.jsonl`

In [ ]:
for cond, topics in TOPICS.items():
    if len(topics) == 0:
        raise ValueError(f"TOPICS['{cond}'] is empty. Add your own topics in Step 3.")


seen = {}
for cond, topics in TOPICS.items():
    for t in topics:
        key = re.sub(r"[^a-z0-9 ]", "", t.lower()).strip()
        if key in seen and seen[key] != cond:
            raise ValueError(f"Topic appears in both {seen[key]} and {cond}: {t!r}")
        seen[key] = cond


conds = list(TOPICS)
for i, c1 in enumerate(conds):
    for c2 in conds[i + 1:]:
        for a in TOPICS[c1]:
            for b in TOPICS[c2]:
                r = SequenceMatcher(None, a.lower(), b.lower()).ratio()
                if r >= 0.7:
                    print(f"WARNING similar topics across conditions ({r:.2f}): [{c1}] {a!r}  ~  [{c2}] {b!r}")


for cond, topics in TOPICS.items():
    pairs = [(a, b) for i, a in enumerate(topics) for b in topics[i + 1:]
             if SequenceMatcher(None, a.lower(), b.lower()).ratio() >= 0.85]
    print(f"condition {cond}: {len(topics)} topics, {len(pairs)} near-duplicate pairs (similarity >= 0.85)")

items = []
for cond, topics in TOPICS.items():
    for t_idx, topic in enumerate(topics):
        for tpl_idx, tpl in enumerate(TEMPLATES):
            items.append({
                "id": f"{cond}{t_idx:03d}_t{tpl_idx}",
                "condition": cond,
                "topic": topic,
                "template": tpl_idx,
                "prompt": tpl.format(n=N_REFS, topic=topic) + FORMAT_INSTRUCTION,
            })

with open("prompts.jsonl", "w") as f:
    for it in items:
        f.write(json.dumps(it, ensure_ascii=False) + "\n")

print(len(items), "prompts written;", Counter(i["condition"] for i in items))
print("Example prompt:\n", items[0]["prompt"])

condition A: 15 topics, 0 near-duplicate pairs (similarity >= 0.85)
condition B: 15 topics, 0 near-duplicate pairs (similarity >= 0.85)
condition C: 15 topics, 0 near-duplicate pairs (similarity >= 0.85)
90 prompts written; Counter({'A': 30, 'B': 30, 'C': 30})
Example prompt:
 List 5 real research papers about: Transformer. For each give title, authors, year, venue, doi and url. Return ONLY a JSON list of objects with the keys title, authors, year, venue, doi, url. No other text.


## Step 5 - Load the model
If the model is gated on Hugging Face, accept its license on the model page and log in first
(uncomment the `login()` lines).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


torch.manual_seed(SEED)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=dtype, device_map="auto", trust_remote_code=TRUST_REMOTE_CODE
)
model.eval()
print("Loaded", MODEL_ID, "in", dtype, "| parameters (B):", round(sum(p.numel() for p in model.parameters()) / 1e9, 2))

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loaded microsoft/Phi-3-mini-4k-instruct in torch.bfloat16 | parameters (B): 3.82


## Step 6 - Generate (greedy decoding, resumable)
Greedy decoding makes the run deterministic. The loop skips items already written, so you can
re-run this cell after a Colab disconnect.

In [ ]:
OUT_PATH = "outputs.jsonl"
done = set()
if os.path.exists(OUT_PATH):
    done = {json.loads(l)["id"] for l in open(OUT_PATH)}

with open(OUT_PATH, "a") as f_out:
    for i, item in enumerate(items):
        if item["id"] in done:
            continue
        
        messages = [{"role": "user", "content": item["prompt"]}]
        inputs = tok.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        )

        
        if isinstance(inputs, dict) or hasattr(inputs, "data"):
            input_ids = inputs["input_ids"]
        else:
            input_ids = inputs


        if not isinstance(input_ids, torch.Tensor):
            input_ids = torch.tensor(input_ids)
        if input_ids.ndim == 1:
            input_ids = input_ids.unsqueeze(0)
        input_ids = input_ids.to(model.device)

        with torch.no_grad():
            out = model.generate(input_ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        text = tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

        rec = dict(item, response=text, model_id=MODEL_ID, decoding="greedy",
                   max_new_tokens=MAX_NEW_TOKENS)
        f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f_out.flush()
        if i % 10 == 0:
            print(f"{i+1}/{len(items)} done")
print("Generation finished.")

1/90 done
11/90 done
21/90 done
31/90 done
41/90 done
51/90 done
61/90 done
71/90 done
81/90 done
Generation finished.


## Step 7 - Verification functions
Each reference gets exactly one label:

| Label | Meaning |
|---|---|
| `REAL_OK` | A matching paper exists in Crossref **or** OpenAlex **and** the supplied DOI/link resolves to it |
| `REAL_BAD_LINK` | The paper exists, but the supplied link/DOI is dead |
| `WRONG_PAPER_LINK` | The DOI or arXiv link is valid but belongs to a **different** paper than the one named |
| `FABRICATED` | No sufficiently similar paper found in Crossref or OpenAlex |
| `UNPARSABLE` | The response contained no valid JSON list (reported separately) |



In [7]:
CROSSREF = "https://api.crossref.org/works"
SIM_THRESHOLD = 0.90
OPENALEX = "https://api.openalex.org/works"
ARXIV_ID = re.compile(r"arxiv\.org/(?:abs|pdf)/(\d{4}\.\d{4,5})")

def norm(s):
    # lowercase and strip punctuation so titles compare fairly
    return re.sub(r"[^a-z0-9 ]", "", (s or "").lower()).strip()

def title_sim(a, b):
    return SequenceMatcher(None, norm(a), norm(b)).ratio()

def extract_refs(text):
    # drop any <think>...</think> reasoning block (also an unclosed one from truncation)
    text = re.sub(r"<think>.*?(</think>|$)", "", text, flags=re.S)
    # first '[' to last ']' ; return [] if the JSON is malformed
    m = re.search(r"\[.*\]", text, flags=re.S)
    if not m:
        return []
    try:
        data = json.loads(m.group(0))
        return [r for r in data if isinstance(r, dict)]
    except json.JSONDecodeError:
        return []

@lru_cache(maxsize=None)
def crossref_title(doi):
    # title Crossref holds for this DOI, or None if the DOI does not exist
    try:
        r = requests.get(f"{CROSSREF}/{doi}", params={"mailto": MAILTO}, timeout=20)
        if r.status_code != 200:
            return None
        t = r.json()["message"].get("title", [])
        return t[0] if t else None
    except Exception:
        return None

@lru_cache(maxsize=None)
def crossref_search(title):
    # best Crossref match (title, doi) for a free-text title
    try:
        r = requests.get(CROSSREF, params={"query.bibliographic": title, "rows": 1,
                                           "mailto": MAILTO}, timeout=20)
        it = r.json()["message"]["items"]
        if not it or not it[0].get("title"):
            return (None, None)
        return (it[0]["title"][0], it[0].get("DOI"))
    except Exception:
        return (None, None)

@lru_cache(maxsize=None)
def link_resolves(url):
    # alive = HTTP < 400 after redirects; 401/403/429 mean "server answered but blocks bots/paywall"
    if not url:
        return False
    try:
        r = requests.get(url, timeout=20, allow_redirects=True,
                         headers={"User-Agent": "ref-verifier/0.1"})
        return r.status_code < 400 or r.status_code in (401, 403, 429)
    except Exception:
        return False

@lru_cache(maxsize=None)
def openalex_search(title):
    # second index: covers arXiv and conference papers that Crossref often lacks
    try:
        r = requests.get(OPENALEX, params={"search": title, "per-page": 1, "mailto": MAILTO}, timeout=20)
        res = r.json().get("results", [])
        return (res[0].get("display_name") or res[0].get("title")) if res else None
    except Exception:
        return None

@lru_cache(maxsize=None)
def arxiv_title(url):
    # if the model gave an arXiv link, fetch the title that arXiv holds for that ID
    m = ARXIV_ID.search(url or "")
    if not m:
        return None
    time.sleep(3)   # arXiv asks for at most one request every 3 seconds
    try:
        r = requests.get("https://export.arxiv.org/api/query", params={"id_list": m.group(1)}, timeout=20)
        titles = re.findall(r"<title>(.*?)</title>", r.text, flags=re.S)
        return re.sub(r"\s+", " ", titles[1]).strip() if len(titles) > 1 else None
    except Exception:
        return None

def classify(ref):
    title = str(ref.get("title") or "")
    doi = (ref.get("doi") or "").strip() or None
    url = (ref.get("url") or "").strip() or None

    # Does the paper exist in at least one index?
    cr_title, cr_doi = crossref_search(title) if title else (None, None)
    oa_title = openalex_search(title) if title else None
    exists = any(t is not None and title_sim(title, t) >= SIM_THRESHOLD for t in (cr_title, oa_title))

    # What does the model's OWN link really point to? (DOI first, else an arXiv ID)
    linked_title = crossref_title(doi) if doi else None
    if linked_title is None:
        linked_title = arxiv_title(url)
    link_matches = linked_title is not None and title_sim(title, linked_title) >= SIM_THRESHOLD
    link_ok = link_resolves(url) or (link_resolves(f"https://doi.org/{doi}") if doi else False)

    if linked_title is not None and not link_matches:
        label = "WRONG_PAPER_LINK"         # valid DOI/arXiv ID, different paper
    elif exists and (link_matches or link_ok):
        label = "REAL_OK"
    elif exists:
        label = "REAL_BAD_LINK"
    else:
        label = "FABRICATED"
    return label, {"crossref_match": cr_title, "openalex_match": oa_title,
                   "link_points_to": linked_title, "link_ok": link_ok}

# quick offline sanity check of the helpers
assert title_sim("Attention Is All You Need", "attention is all you need.") == 1.0
assert extract_refs('text [{"title": "A"}] more') == [{"title": "A"}]
print("helpers OK")

helpers OK


## Step 8 - Verify every reference (takes a while; saves progress)

In [8]:
rows = []
for line in open("outputs.jsonl"):
    rec = json.loads(line)
    refs = extract_refs(rec["response"])
    if not refs:
        # keep unparsable responses visible instead of silently dropping them
        rows.append({"id": rec["id"], "condition": rec["condition"], "topic": rec["topic"],
                     "title": "", "label": "UNPARSABLE"})
        continue
    for ref in refs:
        label, info = classify(ref)
        rows.append({"id": rec["id"], "condition": rec["condition"], "topic": rec["topic"],
                     "title": ref.get("title"), "claimed_doi": ref.get("doi"),
                     "claimed_url": ref.get("url"), "label": label, **info})
        time.sleep(0.1)   # be polite to the APIs

df = pd.DataFrame(rows)
df.to_csv("verified.csv", index=False)
print(len(df), "rows saved to verified.csv")
df["label"].value_counts()

438 rows saved to verified.csv


,count
label,
FABRICATED,290
WRONG_PAPER_LINK,63
REAL_OK,60
REAL_BAD_LINK,22
UNPARSABLE,3


## Step 9 - Statistics
* Rates use **parsed references** as the denominator; unparsable responses are reported separately.
* Each rate has a **Wilson 95% interval**.
* Reference-level **Fisher's exact test** on `REAL_OK` vs not. Caveat: references from the same or similar topics are not independent, so this test is *anti-conservative* (p-values too small).
* The **topic-level analysis** (next cell) treats the topic as the unit of replication and is the primary result.

In [9]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (c - h, c + h)

parsed = df[df["label"] != "UNPARSABLE"]
labels = ["REAL_OK", "REAL_BAD_LINK", "WRONG_PAPER_LINK", "FABRICATED"]

print("Unparsable responses per condition:")
print(df[df["label"] == "UNPARSABLE"].groupby("condition").size(), "\n")

table = []
for cond in sorted(parsed["condition"].unique()):
    sub = parsed[parsed["condition"] == cond]
    n = len(sub)
    row = {"condition": cond, "n_refs": n}
    for lab in labels:
        k = int((sub["label"] == lab).sum())
        lo, hi = wilson_ci(k, n)
        row[lab] = f"{k}/{n} = {k/n:.2f} [{lo:.2f}, {hi:.2f}]"
    table.append(row)
summary = pd.DataFrame(table)
print(summary.to_string(index=False))

def compare(c1, c2):
    a = parsed[parsed["condition"] == c1]["label"] == "REAL_OK"
    b = parsed[parsed["condition"] == c2]["label"] == "REAL_OK"
    tbl = [[int(a.sum()), int((~a).sum())], [int(b.sum()), int((~b).sum())]]
    odds, p = fisher_exact(tbl)
    print(f"REAL_OK {c1} vs {c2}: {tbl[0]} vs {tbl[1]}  Fisher exact p = {p:.4g}")

for pair in [("A", "B"), ("A", "C"), ("B", "C")]:
    if all(c in parsed["condition"].unique() for c in pair):
        compare(*pair)

Unparsable responses per condition:
condition
A    1
C    2
dtype: int64 

condition  n_refs                    REAL_OK              REAL_BAD_LINK           WRONG_PAPER_LINK                  FABRICATED
        A     145 56/145 = 0.39 [0.31, 0.47] 22/145 = 0.15 [0.10, 0.22] 44/145 = 0.30 [0.23, 0.38]  23/145 = 0.16 [0.11, 0.23]
        B     150  4/150 = 0.03 [0.01, 0.07]  0/150 = 0.00 [0.00, 0.02] 10/150 = 0.07 [0.04, 0.12] 136/150 = 0.91 [0.85, 0.94]
        C     140  0/140 = 0.00 [0.00, 0.03]  0/140 = 0.00 [0.00, 0.03]  9/140 = 0.06 [0.03, 0.12] 131/140 = 0.94 [0.88, 0.97]
REAL_OK A vs B: [56, 89] vs [4, 146]  Fisher exact p = 7.071e-16
REAL_OK A vs C: [56, 89] vs [0, 140]  Fisher exact p = 5.852e-20
REAL_OK B vs C: [4, 146] vs [0, 140]  Fisher exact p = 0.1233


### Step 9b - Topic-level analysis (primary result)
Each topic gets one number: its share of `REAL_OK` references (averaged over templates).
Conditions are then compared on those topic-level numbers with a Mann-Whitney U test, and each
condition mean gets a **cluster bootstrap** interval (resampling topics, not references).

In [10]:
import numpy as np
from scipy.stats import mannwhitneyu

# One row per topic: fraction of its parsed references labelled REAL_OK
topic_rate = (parsed.assign(ok=(parsed["label"] == "REAL_OK"))
                    .groupby(["condition", "topic"])["ok"].mean().reset_index())

rng = np.random.default_rng(SEED)
def boot_ci(vals, reps=5000):
    vals = np.asarray(vals, dtype=float)
    means = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(reps)]
    return np.percentile(means, [2.5, 97.5])

for cond, g in topic_rate.groupby("condition"):
    lo, hi = boot_ci(g["ok"])
    print(f"condition {cond}: {len(g)} topics, mean REAL_OK rate {g['ok'].mean():.2f} "
          f"(topic-bootstrap 95% CI {lo:.2f}-{hi:.2f})")

for c1, c2 in [("A", "B"), ("A", "C"), ("B", "C")]:
    a = topic_rate[topic_rate["condition"] == c1]["ok"]
    b = topic_rate[topic_rate["condition"] == c2]["ok"]
    if len(a) and len(b):
        u, p = mannwhitneyu(a, b, alternative="two-sided")
        print(f"{c1} vs {c2}: Mann-Whitney U = {u:.1f}, p = {p:.4g}")
topic_rate.to_csv("topic_level_rates.csv", index=False)

condition A: 15 topics, mean REAL_OK rate 0.38 (topic-bootstrap 95% CI 0.23-0.54)
condition B: 15 topics, mean REAL_OK rate 0.03 (topic-bootstrap 95% CI 0.00-0.07)
condition C: 15 topics, mean REAL_OK rate 0.00 (topic-bootstrap 95% CI 0.00-0.00)
A vs B: Mann-Whitney U = 201.5, p = 8.448e-05
A vs C: Mann-Whitney U = 210.0, p = 8.456e-06
B vs C: Mann-Whitney U = 127.5, p = 0.1644
